In [ ]:
# -*- coding: utf-8 -*-
"""DATA CLEANING.ipynb

Automatically generated by Colab.

Original file is located at
    https://colab.research.google.com/drive/1mf0uy5TpHAPsCCss4WUVHlnVCqobIfg2
"""

## PROJET FIN D'ÉTUDES - INGÉNIEUR IA
PHASE 1 : DATA CLEANING
Auteur : [Votre Nom]
Date   : Février 2026

In [ ]:
import pandas as pd
import numpy as np
import json
import hashlib
from datetime import date
import warnings
warnings.filterwarnings('ignore')

print("=" * 60)
print("PHASE 1 - DATA CLEANING")
print("=" * 60)

## BLOC 1 — CHARGEMENT DES 4 FICHIERS

In [ ]:
print("\n📂 BLOC 1 : Chargement des fichiers...")

# --- Fichier 1 : CSV 2024 ---
df_2024 = pd.read_csv(
    "extractions_taches_2024.csv",
    dtype=str,          # On charge TOUT en string pour éviter les erreurs de type
    encoding="utf-8"
)
print(f"  ✅ CSV 2024     : {len(df_2024)} lignes | {df_2024.shape[1]} colonnes")

# --- Fichier 2 : XLSX 2025 ---
# On ignore le 2ème onglet "Récap_Auto" qui est du bruit
df_2025 = pd.read_excel(
    "extractions_taches_2025.xlsx",
    sheet_name="Taches_2025",
    dtype=str,
    header=0
)
# Supprimer la ligne vide insérée au milieu du fichier
df_2025 = df_2025.dropna(how='all')
print(f"  ✅ XLSX 2025    : {len(df_2025)} lignes | {df_2025.shape[1]} colonnes")

# --- Fichier 3 : JSON agents ---
with open("agents_equipes.json", "r", encoding="utf-8") as f:
    agents_raw = json.load(f)
df_agents = pd.DataFrame(agents_raw)
print(f"  ✅ JSON agents  : {len(df_agents)} lignes | {df_agents.shape[1]} colonnes")

# --- Fichier 4 : CSV absences ---
df_absences = pd.read_csv(
    "absences_conges.csv",
    dtype=str,
    encoding="utf-8"
)
print(f"  ✅ CSV absences : {len(df_absences)} lignes | {df_absences.shape[1]} colonnes")

## BLOC 2 — HARMONISATION DES COLONNES AVANT FUSION

In [ ]:
print("\n🔧 BLOC 2 : Harmonisation des colonnes...")

# Le fichier XLSX 2025 a des noms de colonnes différents du CSV 2024
# On les renomme pour qu'ils soient identiques
rename_2025 = {
    "id tache":              "ID_Tache",
    "MATRICULE AGENT":       "Matricule_Agent",
    "type contrat":          "Type_Contrat",
    "Service ":              "Service",       # ← espace en trop !
    "type_processus":        "Type_Processus",
    "date creation":         "Date_Creation",
    "Date Prise En Charge":  "Date_Prise_En_Charge",
    "DATE_CLOTURE":          "Date_Cloture",
    "tps_passe (min)":       "Temps_Passe_Declare_Min",
    "Nb Dossiers":           "Volume_Dossiers",
    "STATUT":                "Statut",
    "complexite":            "Complexite",
    "commentaires":          "Commentaire"
}
df_2025 = df_2025.rename(columns=rename_2025)

# Ajouter une colonne source pour traçabilité
df_2024["source_fichier"] = "CSV_2024"
df_2025["source_fichier"] = "XLSX_2025"

# Fusion verticale des deux fichiers de tâches
df_taches = pd.concat([df_2024, df_2025], ignore_index=True)
print(f"  ✅ Fusion 2024 + 2025 : {len(df_taches)} lignes")

## BLOC 3 — SUPPRESSION DES DOUBLONS

In [ ]:
print("\n🔍 BLOC 3 : Suppression des doublons...")

avant = len(df_taches)
df_taches = df_taches.drop_duplicates(subset=["ID_Tache"], keep="first")
apres = len(df_taches)
print(f"  ✅ Tâches   : {avant - apres} doublons supprimés → {apres} lignes restantes")

avant_abs = len(df_absences)
df_absences = df_absences.drop_duplicates(keep="first")
apres_abs = len(df_absences)
print(f"  ✅ Absences : {avant_abs - apres_abs} doublons supprimés → {apres_abs} lignes restantes")

avant_ag = len(df_agents)
df_agents = df_agents.drop_duplicates(subset=["matricule"], keep="first")
apres_ag = len(df_agents)
print(f"  ✅ Agents   : {avant_ag - apres_ag} doublons supprimés → {apres_ag} lignes restantes")

## BLOC 4 — NETTOYAGE DES VALEURS MANQUANTES

In [ ]:
print("\n🧹 BLOC 4 : Valeurs manquantes...")

# Remplacer les chaînes vides et "None" par NaN réel
df_taches  = df_taches.replace({"": np.nan, "None": np.nan, "nan": np.nan})
df_agents  = df_agents.replace({"": np.nan, "None": np.nan, "nan": np.nan})
df_absences = df_absences.replace({"": np.nan, "None": np.nan, "nan": np.nan})

# Rapport des valeurs manquantes sur les colonnes clés
colonnes_cles = ["ID_Tache", "Matricule_Agent", "Date_Creation", "Date_Cloture",
                 "Temps_Passe_Declare_Min", "Volume_Dossiers", "Statut", "Type_Contrat"]
print("\n  Taux de valeurs manquantes (colonnes clés) :")
for col in colonnes_cles:
    if col in df_taches.columns:
        taux = df_taches[col].isna().mean() * 100
        flag = "⚠️ " if taux > 10 else "  "
        print(f"  {flag} {col:<35} : {taux:.1f}%")

# Supprimer les lignes sans ID_Tache (inutilisables)
avant = len(df_taches)
df_taches = df_taches.dropna(subset=["ID_Tache"])
print(f"\n  ✅ Lignes sans ID_Tache supprimées : {avant - len(df_taches)}")

# Supprimer les lignes sans Matricule_Agent (on ne sait pas qui a traité)
avant = len(df_taches)
df_taches = df_taches.dropna(subset=["Matricule_Agent"])
print(f"  ✅ Lignes sans Matricule supprimées : {avant - len(df_taches)}")

## BLOC 5 — HARMONISATION DES DATES (6 formats différents)

In [ ]:
print("\n📅 BLOC 5 : Harmonisation des dates...")

FORMATS_DATE_CONNUS = ["%d/%m/%Y", "%Y-%m-%d", "%d.%m.%Y", "%d-%m-%Y", "%m/%d/%Y", "%d/%m/%y"]

def parser_date(serie):
    """
    Parse une série de dates contenant plusieurs formats mélangés.
    Essaie successivement chaque format connu (vectorisé, donc rapide
    même sur de gros volumes), contrairement à pd.to_datetime() seul
    qui (avec pandas >= 2.x) infère un seul format pour toute la colonne
    et génère massivement des NaT sur des colonnes à formats hétérogènes.
    Les dates non parsables avec aucun format connu deviennent NaT.
    """
    result = pd.Series(pd.NaT, index=serie.index, dtype='datetime64[ns]')
    remaining = serie.notna()
    for fmt in FORMATS_DATE_CONNUS:
        if not remaining.any():
            break
        parsed = pd.to_datetime(serie[remaining], format=fmt, errors='coerce')
        ok = parsed.notna()
        idx = remaining[remaining].index[ok.values]
        result.loc[idx] = parsed[ok].values
        remaining.loc[idx] = False
    return result

for col in ["Date_Creation", "Date_Prise_En_Charge", "Date_Cloture"]:
    avant_nats = df_taches[col].isna().sum()
    df_taches[col] = parser_date(df_taches[col])
    apres_nats = df_taches[col].isna().sum()
    nouvelles_nats = apres_nats - avant_nats
    print(f"  ✅ {col:<30} : {nouvelles_nats} dates non parsables → NaT")

# Même traitement pour les agents et absences
for col in ["date_entree", "date_fin_contrat"]:
    if col in df_agents.columns:
        df_agents[col] = parser_date(df_agents[col])

for col in ["Date_Debut", "Date_Fin"]:
    if col in df_absences.columns:
        df_absences[col] = parser_date(df_absences[col])

print("  ✅ Toutes les dates sont maintenant au format datetime")

## BLOC 6 — NORMALISATION DES VARIABLES CATÉGORIELLES

In [ ]:
print("\n🏷️  BLOC 6 : Normalisation des catégories...")

# --- Statuts : "CLOTURE", "cloture", "Fermé" → "Clôturé"
mapping_statuts = {
    "CLOTURE": "Clôturé", "cloture": "Clôturé", "Clôturé": "Clôturé",
    "Fermé": "Clôturé", "EN_COURS": "En cours", "En cours": "En cours",
    "Suspendu": "Suspendu"
}
df_taches["Statut"] = df_taches["Statut"].str.strip().map(mapping_statuts).fillna("Inconnu")
print(f"  ✅ Statuts normalisés : {df_taches['Statut'].value_counts().to_dict()}")

# --- Complexité
mapping_complexite = {
    "Simple": "Simple", "simple": "Simple",
    "Moyen": "Moyen",   "moyen": "Moyen",
    "Complexe": "Complexe", "COMPLEXE": "Complexe"
}
df_taches["Complexite"] = df_taches["Complexite"].str.strip().map(mapping_complexite).fillna("Non renseigné")

# --- Type de contrat
mapping_contrat = {
    "CDI": "CDI", "cdi": "CDI",
    "CDD": "CDD", "cdd": "CDD",
    "Intérimaire": "Intérimaire", "Interimaire": "Intérimaire", "INTERIMAIRE": "Intérimaire",
    "Alternant": "Alternant", "alternant": "Alternant",
    "Prestataire": "Prestataire", "PRESTATAIRE": "Prestataire",
    "Temps partiel": "Temps partiel"
}
df_taches["Type_Contrat"] = df_taches["Type_Contrat"].str.strip().map(mapping_contrat).fillna("Non renseigné")
df_agents["type_contrat"] = df_agents["type_contrat"].str.strip().map(mapping_contrat).fillna("Non renseigné")
print(f"  ✅ Types contrat normalisés")

# --- Absences
mapping_absences = {
    "CP": "Congé Payé", "Congé Payé": "Congé Payé", "CONGE_PAYE": "Congé Payé",
    "Maladie": "Maladie", "MALADIE": "Maladie",
    "RTT": "RTT", "rtt": "RTT",
    "Formation": "Formation", "FORMATION": "Formation",
    "Absent": "Absence non justifiée",
    "Semaine école": "Semaine école", "SEMAINE_ECOLE": "Semaine école",
    "Formation CFA": "Semaine école"
}
df_absences["Type_Absence"] = df_absences["Type_Absence"].str.strip().map(mapping_absences).fillna("Autre")
print(f"  ✅ Types absences normalisés : {df_absences['Type_Absence'].value_counts().to_dict()}")

## BLOC 7 — TRAITEMENT DES VALEURS ABERRANTES

In [ ]:
print("\n⚠️  BLOC 7 : Valeurs aberrantes...")

df_taches["Temps_Passe_Declare_Min"] = pd.to_numeric(df_taches["Temps_Passe_Declare_Min"], errors='coerce')
df_taches["Volume_Dossiers"]         = pd.to_numeric(df_taches["Volume_Dossiers"],         errors='coerce')
df_absences["Duree_Jours"]           = pd.to_numeric(df_absences["Duree_Jours"],           errors='coerce')

# Temps négatifs → NaN
nb_temps_neg = (df_taches["Temps_Passe_Declare_Min"] < 0).sum()
df_taches.loc[df_taches["Temps_Passe_Declare_Min"] < 0, "Temps_Passe_Declare_Min"] = np.nan
print(f"  ✅ Temps négatifs mis à NaN       : {nb_temps_neg} valeurs")

# Volumes aberrants (> 500) → NaN
nb_vol_aber = (df_taches["Volume_Dossiers"] > 500).sum()
df_taches.loc[df_taches["Volume_Dossiers"] > 500, "Volume_Dossiers"] = np.nan
print(f"  ✅ Volumes aberrants mis à NaN     : {nb_vol_aber} valeurs")

# Durées d'absence aberrantes (< 0 ou > 90 jours) → NaN
nb_abs_aber = ((df_absences["Duree_Jours"] < 0) | (df_absences["Duree_Jours"] > 90)).sum()
df_absences.loc[(df_absences["Duree_Jours"] < 0) | (df_absences["Duree_Jours"] > 90), "Duree_Jours"] = np.nan
print(f"  ✅ Durées absences aberrantes NaN  : {nb_abs_aber} valeurs")

# Cohérence des dates : Date_Cloture ne peut pas être avant Date_Creation
nb_incoherents = (df_taches["Date_Cloture"] < df_taches["Date_Creation"]).sum()
df_taches.loc[df_taches["Date_Cloture"] < df_taches["Date_Creation"], "Date_Cloture"] = np.nan
print(f"  ✅ Dates incohérentes (clôture < création) : {nb_incoherents} corrigées")

## BLOC 8 — JOINTURE AGENTS ↔ TÂCHES

In [ ]:
print("\n🔗 BLOC 8 : Jointure des datasets...")

# Renommer la colonne matricule dans agents pour la jointure
df_agents_clean = df_agents.rename(columns={
    "matricule":        "Matricule_Agent",
    "service":          "Service_Agent",
    "type_contrat":     "Type_Contrat_Agent",
    "temps_travail":    "Temps_Travail",
    "date_entree":      "Date_Entree",
    "date_fin_contrat": "Date_Fin_Contrat",
    "actif":            "Actif"
})

# Jointure LEFT : on garde toutes les tâches, on enrichit avec les infos agents
df_final = df_taches.merge(
    df_agents_clean[["Matricule_Agent", "Service_Agent", "Type_Contrat_Agent",
                     "Temps_Travail", "Date_Entree", "Date_Fin_Contrat", "Actif"]],
    on="Matricule_Agent",
    how="left"
)

# Agents fantômes : dans les tâches mais pas dans le référentiel
nb_fantomes = df_final["Service_Agent"].isna().sum()
print(f"  ✅ Tâches sans agent référencé (fantômes) : {nb_fantomes}")
print(f"  ✅ Dataset final après jointure : {len(df_final)} lignes | {df_final.shape[1]} colonnes")

## BLOC 9 — EXPORT DU DATASET PROPRE

In [ ]:
print("\n💾 BLOC 9 : Export...")

df_final.to_csv("dataset_clean.csv", index=False, encoding="utf-8")
df_absences.to_csv("absences_clean.csv", index=False, encoding="utf-8")

print(f"  ✅ dataset_clean.csv   exporté → {len(df_final)} lignes")
print(f"  ✅ absences_clean.csv  exporté → {len(df_absences)} lignes")

## RAPPORT FINAL

In [ ]:
print("\n" + "=" * 60)
print("RAPPORT FINAL - DATASET PROPRE")
print("=" * 60)
print(f"  Lignes totales             : {len(df_final)}")
print(f"  Colonnes                   : {df_final.shape[1]}")
print(f"  Période couverte           : {df_final['Date_Creation'].min().date()} → {df_final['Date_Creation'].max().date()}")
print(f"  Agents uniques             : {df_final['Matricule_Agent'].nunique()}")
print(f"  Services uniques           : {df_final['Service'].nunique()}")
print(f"\n  Répartition types contrat :")
print(df_final['Type_Contrat'].value_counts().to_string())
print(f"\n  Taux de complétion Date_Cloture : {df_final['Date_Cloture'].notna().mean()*100:.1f}%")
print(f"  Taux de complétion Temps_Passé  : {df_final['Temps_Passe_Declare_Min'].notna().mean()*100:.1f}%")
print("\n✅ Phase 1 terminée ! Prêt pour le Feature Engineering.")